[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajeevraibhatia/agent-harness-evals/blob/main/building-agents/notebooks/m4_research_bot.ipynb) [![Course](https://img.shields.io/badge/Course-rajeevraibhatia.com-7c3aed)](https://rajeevraibhatia.com/curriculum/building-agents#module-4)

# Module 4 — Mini-Project: Research Bot

**Level:** Beginner | **Time:** ~1h | [Full reading →](https://rajeevraibhatia.com/curriculum/building-agents#module-4)

### What you'll build
A complete research bot: search → save findings → synthesize answer.

### The three-tool research pattern
1. `search(query)` — find information
2. `save_finding(source, text)` — store what matters
3. `write_summary(question)` — synthesize the answer

This exact pattern is in production at every major AI company.

In [ ]:
!pip install openai --quiet
# Optional for real search: !pip install tavily-python --quiet

In [ ]:
import json
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MAX_STEPS = 15
_findings: list[dict] = []

MOCK_SEARCH_DB = {
    "transformer": "Transformers use self-attention (Vaswani 2017). Input → embeddings → N × (MHA + FFN) → output.",
    "attention": "Attention = softmax(QK^T / sqrt(d_k)) × V. Multi-head runs h parallel heads.",
    "bert": "BERT (Devlin 2018): bidirectional, pretrained with MLM. Fine-tune AdamW, lr=2e-5.",
    "rag": "RAG (Lewis 2020): retrieve docs, condition generation. Reduces hallucinations.",
}

TOOLS = [
    {"type": "function", "function": {
        "name": "search",
        "description": "Search for information. Be specific — 'BERT fine-tuning 2024' beats 'BERT'.",
        "parameters": {"type": "object",
            "properties": {"query": {"type": "string"}}, "required": ["query"]}
    }},
    {"type": "function", "function": {
        "name": "save_finding",
        "description": "Store a relevant passage. Call after each useful search.",
        "parameters": {"type": "object",
            "properties": {"source": {"type": "string"}, "text": {"type": "string"}},
            "required": ["source", "text"]}
    }},
    {"type": "function", "function": {
        "name": "write_summary",
        "description": "Write a final answer from saved findings. Call only after saving 2+ findings.",
        "parameters": {"type": "object",
            "properties": {"question": {"type": "string"}}, "required": ["question"]}
    }},
]

def run_tool(name, args):
    if name == "search":
        q = args["query"].lower()
        hits = [v for k, v in MOCK_SEARCH_DB.items() if k in q]
        return hits[0] if hits else "No results found."
    if name == "save_finding":
        _findings.append(args); return f"Saved finding #{len(_findings)}"
    if name == "write_summary":
        if not _findings: return "No findings yet. Search first."
        context = "\n".join(f"[{f['source']}] {f['text']}" for f in _findings)
        r = client.chat.completions.create(model="gpt-4o", messages=[
            {"role": "system", "content": "Synthesize these findings into a clear answer."},
            {"role": "user", "content": f"Question: {args['question']}\n\nFindings:\n{context}"}
        ])
        return r.choices[0].message.content
    return f"Unknown tool: {name}"

def research(question: str) -> str:
    _findings.clear()
    messages = [
        {"role": "system", "content": "Research thoroughly: search 2-3 times, save findings, then write_summary."},
        {"role": "user", "content": question}
    ]
    for _ in range(MAX_STEPS):
        r = client.chat.completions.create(model="gpt-4o", messages=messages, tools=TOOLS)
        msg = r.choices[0].message
        messages.append(msg)
        if not msg.tool_calls: return msg.content
        for tc in msg.tool_calls:
            result = run_tool(tc.function.name, json.loads(tc.function.arguments))
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return "Max steps reached."

answer = research("How does the transformer attention mechanism work?")
print(answer)

## Exercise

(a) Replace mock search with Tavily. (b) Add relevance score filtering.

In [ ]:
# Capstone Exercise
# (a) Replace MOCK_SEARCH_DB with Tavily real search:
#     pip install tavily-python
#     from tavily import TavilyClient
#     client_tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
#
# (b) Add relevance_score: float to save_finding schema.
#     Only save findings with score >= 0.6.
#     Return "Skipped — score below threshold" for low-score findings.

# Your code here:

## What's Next?

You've completed the Building Agents mini-course. The advanced course goes deeper: production harnesses, multi-agent systems, eval suites, and safety.

[**Building an Agent Harness with Evals →**](https://rajeevraibhatia.com/curriculum/agent-harness-evals)

In [ ]:
# 🎉 You built a working research agent!
#
# You now know:
# ✓ Tool calling and the agent loop
# ✓ Multi-tool agents with memory
# ✓ Reliability: max steps, error wrapping, circuit breaker, retries
# ✓ A real end-to-end research pipeline
#
# Ready for the next level?
# Building an Agent Harness with Evals:
# https://rajeevraibhatia.com/curriculum/agent-harness-evals
print("Course complete! Next: https://rajeevraibhatia.com/curriculum/agent-harness-evals")